In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
#import geobr
import cartopy.crs as ccrs
import cartopy.feature as cfeature
#from cartopy.feature import NaturalEarthFeature
from cmcrameri import cm
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

In [ ]:
shapefile_path = os.path.join("marajo_basin_shapefile","marajo_basin.shp")
marajo = gpd.read_file(shapefile_path)
# Convert to geographic coordinates if necessary
if marajo.crs is None:
    raise ValueError("The shapefile has no CRS information.")
marajo = marajo.to_crs(epsg=4326)
print("Shapefile loaded successfully.")
print(f"CRS: {marajo.crs}")
print(f"Number of features: {len(marajo)}")
print("Bounds [lon_min, lat_min, lon_max, lat_max]:",marajo.total_bounds)

In [ ]:
data = np.loadtxt("marajo_data.xyz", skiprows=1, delimiter=",")
longitude = data[:, 0]
latitude = data[:, 1]
gravity = data[:, 4]

n_points = data.shape[0]

# Determine the number of unique coordinates
lon_unique = np.unique(longitude)
lat_unique = np.unique(latitude)

n_lon = lon_unique.size
n_lat = lat_unique.size

print(f"Number of rows: {n_points}")
print(f"Unique longitudes: {n_lon}")
print(f"Unique latitudes:  {n_lat}")
print(f"Expected regular-grid size: {n_lat} x {n_lon}")
print(f"Expected number of points: {n_lat * n_lon}")

if n_lat * n_lon != n_points:
    raise ValueError("The number of rows is incompatible with a complete regular grid. "
                     "Check for missing points, duplicates, or irregular sampling.")
# Sort first by latitude and then by longitude before reshaping
order = np.lexsort((longitude, latitude))

lon_sorted = longitude[order]
lat_sorted = latitude[order]
gravity_sorted = gravity[order]

LON = lon_sorted.reshape(n_lat, n_lon)
LAT = lat_sorted.reshape(n_lat, n_lon)
GRAV = gravity_sorted.reshape(n_lat, n_lon)

In [ ]:
fig = plt.figure(figsize=(8,7))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.set_extent([LON.min(), LON.max(), LAT.min(), LAT.max()],crs=ccrs.PlateCarree())
cf = ax.contourf(LON,LAT,GRAV,levels=50,cmap=cm.roma_r,extend="both",transform=ccrs.PlateCarree())
marajo.boundary.plot(ax=ax,edgecolor="red",linewidth=1.5,transform=ccrs.PlateCarree(),zorder=10)
ax.add_feature(cfeature.COASTLINE.with_scale("10m"),linewidth=0.7,zorder=8)
gl = ax.gridlines(crs=ccrs.PlateCarree(),draw_labels=True,linewidth=0.5,linestyle="--",alpha=0.5)
gl.top_labels = True
gl.bottom_labels = False
gl.left_labels = True
gl.right_labels = False
gl.x_inline = False
gl.y_inline = False
ticks = np.linspace(GRAV.min(), GRAV.max(),6)
cbar = plt.colorbar(cf,ax=ax,orientation="horizontal",pad=0.01,shrink=0.50,ticks=ticks)
cbar.ax.set_xticklabels([f"{value:.0f}" for value in ticks])
cbar.set_label("Gravity field (mGal)",fontsize=11)
plt.tight_layout()
legend_handles = [Line2D([0], [0],color="r", markerfacecolor="red",markeredgecolor="black", markersize=12, label="Marajó basin")]
ax.legend(handles=legend_handles,loc="lower right",frameon=True,framealpha=0.95,facecolor="white",edgecolor="gray",fontsize=10)
#plt.savefig("results/anomaly.png",dpi=300,bbox_inches="tight")
plt.show()

In [ ]:
shape_files = {
    "Terrestrial structures":r"estruturas/MGBrasil_estruturas_terrestres_escala_1_2500000.shp",
    "Geophysical structures":r"estruturas/MGBrasil_estruturas_terrestres_geofisicas_escala_1_2500000.shp",
    "Continental structures":r"geologia/estruturas_continente.shp",
    "Oceanic structures":r"geologia/estruturas_oceano.shp",
    "Dikes":r"geologia/diques_pa.shp",
    "Structural PA":r"geologia/estrutural_pa.shp"
}

structures = {}
for name,path in shape_files.items():
    gdf = gpd.read_file(path)
    if gdf.crs is None:
        print(f"WARNING: {name} has no CRS.")
    else:
        gdf = gdf.to_crs(epsg=4326)
    structures[name] = gdf
    print("\n","="*70)
    print(name)
    print("="*70)
    print("File:",path)
    print("Features:",len(gdf))
    print("CRS:",gdf.crs)
    print("Geometry:",gdf.geom_type.value_counts().to_dict())
    print("Bounds:",gdf.total_bounds)
    print("Columns:",list(gdf.columns))
    display(gdf.head())

In [ ]:
colors = ["blue","green","magenta","orange","cyan","brown"]

fig,axes = plt.subplots(2,3,figsize=(18,11),subplot_kw={"projection":ccrs.PlateCarree()})

for ax,(name,gdf),color in zip(axes.ravel(),structures.items(),colors):
    ax.set_extent([LON.min(),LON.max(),LAT.min(),LAT.max()],crs=ccrs.PlateCarree())
    gdf.plot(ax=ax,color="none" if "Polygon" in gdf.geom_type.unique() else color,edgecolor=color,linewidth=1.0,zorder=10)
    marajo.boundary.plot(ax=ax,edgecolor="red",linewidth=1.5,zorder=12)
    ax.add_feature(cfeature.COASTLINE.with_scale("10m"),linewidth=0.6,zorder=8)
    gl = ax.gridlines(crs=ccrs.PlateCarree(),draw_labels=True,linewidth=0.4,linestyle="--",alpha=0.4)
    gl.top_labels = False
    gl.bottom_labels = True
    gl.left_labels = True
    gl.right_labels = False
    gl.x_inline = False
    gl.y_inline = False
    ax.set_title(name,fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
shape_colors = {
    "Terrestrial structures":"blue",
    "Geophysical structures":"green",
    "Continental structures":"magenta",
    "Oceanic structures":"orange",
    "Dikes":"cyan",
    "Structural PA":"brown"
}

fig = plt.figure(figsize=(9,8))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.set_extent([LON.min(),LON.max(),LAT.min(),LAT.max()],crs=ccrs.PlateCarree())

cf = ax.contourf(LON,LAT,GRAV,levels=50,cmap=cm.roma_r,extend="both",transform=ccrs.PlateCarree())

for name,gdf in structures.items():
    color = shape_colors[name]
    if any(gdf.geom_type.isin(["Polygon","MultiPolygon"])):
        gdf.boundary.plot(ax=ax,color=color,linewidth=1.0,zorder=11)
    else:
        gdf.plot(ax=ax,color=color,linewidth=1.0,zorder=11)

marajo.boundary.plot(ax=ax,edgecolor="red",linewidth=1.7,zorder=15)
ax.add_feature(cfeature.COASTLINE.with_scale("10m"),linewidth=0.7,zorder=8)

gl = ax.gridlines(crs=ccrs.PlateCarree(),draw_labels=True,linewidth=0.5,linestyle="--",alpha=0.5)
gl.top_labels = True
gl.bottom_labels = False
gl.left_labels = True
gl.right_labels = False
gl.x_inline = False
gl.y_inline = False

ticks = np.linspace(GRAV.min(), GRAV.max(),6)
cbar = plt.colorbar(cf,ax=ax,orientation="horizontal",pad=0.01,shrink=0.50,ticks=ticks)
cbar.ax.set_xticklabels([f"{value:.0f}" for value in ticks])
cbar.set_label("Gravity field (mGal)",fontsize=11)

legend_handles = [
    Line2D([0],[0],color="red",linewidth=2.0,label="Marajó Basin"),
    Line2D([0],[0],color="blue",linewidth=2.0,label="Terrestrial structures"),
    Line2D([0],[0],color="green",linewidth=2.0,label="Geophysical structures"),
    Line2D([0],[0],color="magenta",linewidth=2.0,label="Continental structures"),
    Line2D([0],[0],color="orange",linewidth=2.0,label="Oceanic structures"),
    Line2D([0],[0],color="cyan",linewidth=2.0,label="Dikes"),
    Line2D([0],[0],color="brown",linewidth=2.0,label="Structural PA")
]

ax.legend(handles=legend_handles,loc="lower right",frameon=True,framealpha=0.95,facecolor="white",edgecolor="gray",fontsize=8)

plt.tight_layout()
#plt.savefig("results/anomaly_all_structures.png",dpi=300,bbox_inches="tight")
plt.show()

In [ ]:
marajo_clip = marajo.to_crs(epsg=4326)

structures_marajo = {}

for name,gdf in structures.items():
    gdf = gdf.to_crs(marajo_clip.crs)
    clipped = gpd.clip(gdf,marajo_clip)
    structures_marajo[name] = clipped
    print(f"{name}: {len(gdf)} total features -> {len(clipped)} features intersecting Marajó Basin")

In [ ]:
for name,gdf in structures_marajo.items():
    print(name,len(gdf))

In [ ]:
structures_marajo = {}

for name,gdf in structures.items():
    clipped = gpd.clip(gdf,marajo)
    structures_marajo[name] = clipped
    print("\n"+"="*70)
    print(name)
    print("="*70)
    print("Total features:",len(gdf))
    print("Features inside/intersecting Marajó Basin:",len(clipped))
    
    if len(clipped) > 0:
        print("Geometry:",clipped.geom_type.value_counts().to_dict())
        print("Columns:",list(clipped.columns))

In [ ]:
for name,gdf in structures_marajo.items():

    print("\n"+"="*70)
    print(name)
    print("="*70)

    if len(gdf) == 0:
        print("No structures inside Marajó Basin.")
        continue

    for col in ["TIPO_ESTRU","TIPO","ESTRUTURA","NMESTRUTUR","REGIME_TEC","PROVINCIA","LEGENDA"]:
        if col in gdf.columns:
            print(f"\n{col}:")
            print(gdf[col].value_counts(dropna=False).head(30))

In [ ]:
fig = plt.figure(figsize=(9,8))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.set_extent([LON.min(),LON.max(),LAT.min(),LAT.max()],crs=ccrs.PlateCarree())

vmin = np.nanpercentile(GRAV,2)
vmax = np.nanpercentile(GRAV,98)
levels = np.linspace(vmin,vmax,50)

cf = ax.contourf(LON,LAT,GRAV,levels=levels,cmap=cm.roma_r,extend="both",transform=ccrs.PlateCarree())

structures_marajo["Terrestrial structures"].plot(ax=ax,color="black",linewidth=1.3,zorder=11)
structures_marajo["Geophysical structures"].plot(ax=ax,color="lime",linewidth=1.8,zorder=12)
structures_marajo["Continental structures"].plot(ax=ax,color="magenta",linewidth=1.0,zorder=10)
structures_marajo["Oceanic structures"].plot(ax=ax,color="orange",linewidth=1.5,zorder=11)
structures_marajo["Structural PA"].plot(ax=ax,color="cyan",linewidth=1.0,zorder=13)

marajo.boundary.plot(ax=ax,edgecolor="red",linewidth=2.0,zorder=15)

ax.add_feature(cfeature.COASTLINE.with_scale("10m"),linewidth=0.7,zorder=8)

gl = ax.gridlines(crs=ccrs.PlateCarree(),draw_labels=True,linewidth=0.5,linestyle="--",alpha=0.5)
gl.top_labels = True
gl.bottom_labels = False
gl.left_labels = True
gl.right_labels = False
gl.x_inline = False
gl.y_inline = False

ticks = np.linspace(vmin,vmax,6)
cbar = plt.colorbar(cf,ax=ax,orientation="horizontal",pad=0.01,shrink=0.55,ticks=ticks)
cbar.ax.set_xticklabels([f"{value:.0f}" for value in ticks])
cbar.set_label("Gravity field (mGal)",fontsize=11)

legend_handles = [
    Line2D([0],[0],color="red",linewidth=2.0,label="Marajó Basin"),
    Line2D([0],[0],color="yellow",linewidth=2.0,label="Terrestrial structures"),
    Line2D([0],[0],color="lime",linewidth=2.0,label="Geophysical structures"),
    Line2D([0],[0],color="magenta",linewidth=2.0,label="Continental structures"),
    Line2D([0],[0],color="orange",linewidth=2.0,label="Oceanic structures"),
    Line2D([0],[0],color="cyan",linewidth=2.0,label="Structural PA")
]

ax.legend(handles=legend_handles,loc="lower right",frameon=True,framealpha=0.90,facecolor="white",edgecolor="gray",fontsize=8)

plt.tight_layout()
#plt.savefig("results/bouguer_all_structures_marajo.png",dpi=300,bbox_inches="tight")
plt.show()

In [ ]:
fig = plt.figure(figsize=(9,8))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.set_extent([LON.min(),LON.max(),LAT.min(),LAT.max()],crs=ccrs.PlateCarree())

vmin = np.nanpercentile(GRAV,2)
vmax = np.nanpercentile(GRAV,98)
levels = np.linspace(vmin,vmax,50)

cf = ax.contourf(LON,LAT,GRAV,levels=levels,cmap=cm.roma_r,extend="both",transform=ccrs.PlateCarree())

structures_marajo["Terrestrial structures"].plot(ax=ax,color="black",linewidth=1.3,zorder=11)
structures_marajo["Geophysical structures"].plot(ax=ax,color="lime",linewidth=1.8,zorder=12)
structures_marajo["Structural PA"].plot(ax=ax,color="cyan",linewidth=1.0,zorder=13)
structures_marajo["Oceanic structures"].plot(ax=ax,color="orange",linewidth=1.5,zorder=11)

marajo.boundary.plot(ax=ax,edgecolor="red",linewidth=2.0,zorder=15)
ax.add_feature(cfeature.COASTLINE.with_scale("10m"),linewidth=0.7,zorder=8)

gl = ax.gridlines(crs=ccrs.PlateCarree(),draw_labels=True,linewidth=0.5,linestyle="--",alpha=0.5)
gl.top_labels = True
gl.bottom_labels = False
gl.left_labels = True
gl.right_labels = False
gl.x_inline = False
gl.y_inline = False

ticks = np.linspace(vmin,vmax,7)

cbar = plt.colorbar(cf,ax=ax,orientation="horizontal",pad=0.01,shrink=0.55,ticks=ticks)
cbar.ax.set_xticklabels([f"{value:.0f}" for value in ticks])
cbar.set_label("Gravity Anomaly (mGal)",fontsize=11)

legend_handles = [
    Line2D([0],[0],color="red",linewidth=2.0,label="Marajó Basin"),
    Line2D([0],[0],color="black",linewidth=2.0,label="Mapped structures"),
    Line2D([0],[0],color="lime",linewidth=2.0,label="Geophysical lineaments"),
    Line2D([0],[0],color="cyan",linewidth=2.0,label="Structural lineaments"),
    Line2D([0],[0],color="orange",linewidth=2.0,label="Oceanic structures")
]

ax.legend(handles=legend_handles,loc="lower right",frameon=True,framealpha=0.90,facecolor="white",edgecolor="gray",fontsize=8)

plt.tight_layout()
#plt.savefig("results/bouguer_selected_structures_marajo.png",dpi=300,bbox_inches="tight")
plt.show()

In [ ]:
print("\nTERRESTRIAL")
display(structures_marajo["Terrestrial structures"][["TIPO_ESTRU","geometry"]])

print("\nGEOPHYSICAL")
display(structures_marajo["Geophysical structures"][["NMESTRUTUR","TIPO_ESTRU","geometry"]])

print("\nOCEANIC")
display(structures_marajo["Oceanic structures"][["ESTRUTURA","NOME","geometry"]])

print("\nSTRUCTURAL PA — NAMED STRUCTURES")
display(structures_marajo["Structural PA"][["NMESTRUTUR","TIPO","REGIME_TEC","geometry"]].dropna(subset=["NMESTRUTUR"]))

In [ ]:
terrestrial_final = structures_marajo["Terrestrial structures"].copy()
geophysical_final = structures_marajo["Geophysical structures"].copy()
structural_pa_final = structures_marajo["Structural PA"].copy()
oceanic_final = structures_marajo["Oceanic structures"][structures_marajo["Oceanic structures"]["ESTRUTURA"]!="CANAL SUBMARINO"].copy()

print("Terrestrial structures:",len(terrestrial_final))
print("Geophysical structures:",len(geophysical_final))
print("Structural PA:",len(structural_pa_final))
print("Oceanic tectonic structures:",len(oceanic_final))

In [ ]:
fig = plt.figure(figsize=(8,7))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.set_extent([LON.min(),LON.max(),LAT.min(),LAT.max()],crs=ccrs.PlateCarree())

vmin = np.nanpercentile(GRAV,2)
vmax = np.nanpercentile(GRAV,98)
levels = np.linspace(vmin,vmax,50)

cf = ax.contourf(LON,LAT,GRAV,levels=levels,cmap=cm.roma_r,extend="both",transform=ccrs.PlateCarree())

structural_pa_final.plot(ax=ax,color="cyan",linewidth=1.0,zorder=11)
terrestrial_final.plot(ax=ax,color="black",linewidth=1.3,zorder=12)
geophysical_final.plot(ax=ax,color="lime",linewidth=1.8,zorder=13)
oceanic_final.plot(ax=ax,color="orange",linewidth=1.4,zorder=12)

marajo.boundary.plot(ax=ax,edgecolor="red",linewidth=2.0,zorder=15)
ax.add_feature(cfeature.COASTLINE.with_scale("10m"),linewidth=0.7,zorder=8)

gl = ax.gridlines(crs=ccrs.PlateCarree(),draw_labels=True,linewidth=0.5,linestyle="--",alpha=0.5)
gl.top_labels = True
gl.bottom_labels = False
gl.left_labels = True
gl.right_labels = False
gl.x_inline = False
gl.y_inline = False

ticks = np.linspace(vmin,vmax,7)
cbar = plt.colorbar(cf,ax=ax,orientation="horizontal",pad=0.01,shrink=0.55,ticks=ticks)
cbar.ax.set_xticklabels([f"{value:.0f}" for value in ticks])
cbar.set_label("Gravity field (mGal)",fontsize=11)

legend_handles = [
    Line2D([0],[0],color="red",linewidth=2.0,label="Marajó Basin"),
    Line2D([0],[0],color="cyan",linewidth=2.0,label="Structural PA"),
    Line2D([0],[0],color="black",linewidth=2.0,label="Terrestrial structures"),
    Line2D([0],[0],color="lime",linewidth=2.0,label="Geophysical structures"),
    Line2D([0],[0],color="orange",linewidth=2.0,label="Oceanic tectonic structures")
]

ax.legend(handles=legend_handles,loc="lower right",frameon=True,framealpha=0.92,facecolor="white",edgecolor="gray",fontsize=8)

plt.tight_layout()
#plt.savefig("results/bouguer_structures_marajo.png",dpi=300,bbox_inches="tight")
plt.show()

In [ ]:
import os
import pandas as pd
import geopandas as gpd

output_dir = "estrutural_marajo"
os.makedirs(output_dir,exist_ok=True)

g1 = structures_marajo["Terrestrial structures"].copy()
g1["SOURCE"] = "Terrestrial"
g1["TYPE"] = g1["TIPO_ESTRU"]
g1["NAME"] = g1["NMESTRUTUR"]
g1["REGIME"] = None
g1 = g1[["SOURCE","TYPE","NAME","REGIME","geometry"]]

g2 = structures_marajo["Geophysical structures"].copy()
g2["SOURCE"] = "Geophysical"
g2["TYPE"] = g2["TIPO_ESTRU"]
g2["NAME"] = g2["NMESTRUTUR"]
g2["REGIME"] = None
g2 = g2[["SOURCE","TYPE","NAME","REGIME","geometry"]]

g3 = structures_marajo["Continental structures"].copy()
g3["SOURCE"] = "Continental"
g3["TYPE"] = g3["TIPO"]
g3["NAME"] = g3["NOME_DA_ES"]
g3["REGIME"] = None
g3 = g3[["SOURCE","TYPE","NAME","REGIME","geometry"]]

g4 = structures_marajo["Oceanic structures"].copy()
g4["SOURCE"] = "Oceanic"
g4["TYPE"] = g4["ESTRUTURA"]
g4["NAME"] = g4["NOME"]
g4["REGIME"] = None
g4 = g4[["SOURCE","TYPE","NAME","REGIME","geometry"]]

g5 = structures_marajo["Structural PA"].copy()
g5["SOURCE"] = "Structural"
g5["TYPE"] = g5["TIPO"]
g5["NAME"] = g5["NMESTRUTUR"]
g5["REGIME"] = g5["REGIME_TEC"]
g5 = g5[["SOURCE","TYPE","NAME","REGIME","geometry"]]

estrutural_marajo = gpd.GeoDataFrame(pd.concat([g1,g2,g3,g4,g5],ignore_index=True),crs="EPSG:4326")

estrutural_marajo = estrutural_marajo[~estrutural_marajo.geometry.is_empty & estrutural_marajo.geometry.notna()].copy()

output_shp = os.path.join(output_dir,"estrutural_marajo.shp")
estrutural_marajo.to_file(output_shp,driver="ESRI Shapefile",encoding="UTF-8")

print("Shapefile created:",output_shp)
print("Total features:",len(estrutural_marajo))
print()
print(estrutural_marajo["SOURCE"].value_counts())

In [ ]:
import geopandas as gpd
import pandas as pd

estrutural_marajo = gpd.read_file(r"estrutural_marajo/estrutural_marajo.shp")

print("="*70)
print("MARAJÓ STRUCTURAL DATABASE")
print("="*70)
print("Total features:",len(estrutural_marajo))
print("CRS:",estrutural_marajo.crs)
print("Geometry types:",estrutural_marajo.geom_type.value_counts().to_dict())
print("Bounds:",estrutural_marajo.total_bounds)
print("Columns:",list(estrutural_marajo.columns))

print("\n"+"="*70)
print("FEATURES BY SOURCE")
print("="*70)
print(estrutural_marajo["SOURCE"].value_counts(dropna=False))

print("\n"+"="*70)
print("STRUCTURAL TYPES")
print("="*70)
print(estrutural_marajo["TYPE"].value_counts(dropna=False))

print("\n"+"="*70)
print("TECTONIC REGIMES")
print("="*70)
print(estrutural_marajo["REGIME"].value_counts(dropna=False))

print("\n"+"="*70)
print("NAMED STRUCTURES")
print("="*70)
print(estrutural_marajo.loc[estrutural_marajo["NAME"].notna(),["SOURCE","NAME","TYPE","REGIME"]].to_string(index=False))

print("\n"+"="*70)
print("COMPLETE STRUCTURAL TABLE")
print("="*70)
display(estrutural_marajo[["SOURCE","TYPE","NAME","REGIME","geometry"]])